In [1]:
# Save global mean OSDMA8

In [2]:
import os
import xarray as xr
import numpy as np
from utils.utils import lat_weighted_mean
from utils.utils import land_filter
from utils.utils import get_scenario_config

In [15]:
# === Concatenate ensemble members ===
def merge_global_mean_ensembles(VAR_DIR, model, scenario, years, dates, ens_mem, land_mask=False):
    # land_mask=True allows for the land filter function to remove the ocean
    ens = []

    for ens_num in ens_mem:
        print(f"Processing ensemble member {ens_num:02d}")
        year_range = range(years.start, years.stop)

        in_file = f"OSDMA8_BC_{model}_{scenario}_{ens_num:02d}_{dates}.nc"
        in_path = os.path.join(VAR_DIR, in_file)
        da = xr.open_dataarray(in_path)

        da_years = []
        for year in year_range:
            da_year = da.sel(year=year)
            if land_mask is True:
                da_year = land_filter(da_year)

            da_years.append(lat_weighted_mean(da_year))  # Latitude weighted mean

        ens.append(xr.concat(da_years, "time"))

    new_da = xr.concat(ens, dim=xr.DataArray(np.arange(1, len(ens)+1),
                                             dims="ensemble", name="ensemble"))

    return new_da

In [16]:
# === Scenario and path config ===
# Set to whatever scenario and model you want
# Function returns error if not recognised
model = "UKESM1"
scenario = "G6-1.5K"

config = get_scenario_config(model, scenario)
ensemble_members = config["ensemble_members"]
years = config["years"]

OSDMA8_DIR = f"/glade/work/awells/air_quality/{model}/ozone/OSDMA8_BC/"

# Final year is not be complete due to SH Jan-Mar missing
dates = f"{years.start}-{years.stop - 1}"

land_mean = merge_global_mean_ensembles(
    OSDMA8_DIR,
    model,
    scenario,
    years,
    dates,
    ensemble_members,
    land_mask=True)

land_mean.attrs["description"] = ("Global Mean OSDMA8 - scripts by A.F. "
                                  "Wells (2025)")
land_mean.attrs["units"] = "ppb"
land_mean.attrs["scenario"] = scenario
land_mean.attrs["model"] = model

out_file = f"OSDMA8_BC_globalmean_{model}_{scenario}_{dates}.nc"
out_path = os.path.join(OSDMA8_DIR, out_file)
land_mean.to_netcdf(out_path)

Processing ensemble member 01
Processing ensemble member 02
Processing ensemble member 03
